In [ ]:
import re
import pandas as pd

In [ ]:
df = pd.read_csv("generated_summaries.csv")
asli = pd.read_excel("history_data_with_summary.xlsx").dropna(
    subset=["title", "content", "reference_summary"])
kunci = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
df["k"] = df.reference_summary.map(kunci)
asli["k"] = asli.reference_summary.map(kunci)
df = df.merge(asli[["k", "title", "content"]].drop_duplicates("k"), on="k", how="left")
print("Artikel yang tidak ketemu:", df.content.isna().sum())  # harus 0
df["input_text"] = df.title + " " + df.content   # artikel lengkap
MODELS = {"FLAN-T5": "flan_summary", "IndoBART": "bart_summary"}

def kata(teks):
    return re.findall(r"[a-z0-9]+", str(teks).lower())

def angka(teks):
    return re.findall(r"\d[\d.,]*\d|\d", str(teks))

# ---------- 1. TRUNCATION: ringkasan tidak diakhiri . ! ? ----------
def terpotong(ringkasan):
    return not str(ringkasan).strip().endswith((".", "!", "?"))

# ---------- 2. ENTITY DISTORTION: ada angka/tahun yang tidak ada di artikel ----------
def angka_salah(artikel, ringkasan):
    gabung = set(re.sub(r"[.,]", "", n) for n in angka(artikel))
    pecah = set(re.findall(r"\d+", str(artikel)))
    salah = [n for n in angka(ringkasan)
             if re.sub(r"[.,]", "", n) not in gabung
             and not all(p in pecah for p in re.findall(r"\d+", n))]
    return salah

# ---------- 3. Tanda bantu untuk cek manual ----------
def huruf_hilang(artikel, ringkasan):
    """Kata yang huruf pertamanya hilang, mis. 'erempuan' (dari 'perempuan')."""
    kata_artikel = set(kata(artikel))
    potong = {w[1:] for w in kata_artikel if len(w) > 3}
    return sorted({w for w in kata(ringkasan)
                   if w not in kata_artikel and w in potong})

def kata_asing(artikel, ringkasan):
    """Kata di ringkasan yang tidak ada di artikel (calon typo/kata rusak)."""
    kata_artikel = set(kata(artikel))
    return sorted({w for w in kata(ringkasan)
                   if w not in kata_artikel and not w.isdigit() and len(w) > 2})

def kalimat_curiga(artikel, ringkasan, batas=0.5):
    """Kalimat yang < 50% katanya ada di artikel (calon halusinasi)."""
    kata_artikel = set(kata(artikel))
    hasil = []
    for k in re.split(r"(?<=[.!?])\s+", str(ringkasan)):
        w = [x for x in kata(k) if len(x) > 3]
        if len(w) >= 4 and sum(x in kata_artikel for x in w) / len(w) < batas:
            hasil.append(k.strip())
    return hasil

def urutan_tahun_beda(artikel, ringkasan):
    """True jika urutan tahun di ringkasan berbeda dari urutan di artikel."""
    tahun = list(dict.fromkeys(re.findall(r"\b(1[0-9]{3}|20[0-9]{2})\b", str(ringkasan))))
    posisi = []
    for t in tahun:
        m = re.search(r"\b%s\b" % t, str(artikel))
        if m:
            posisi.append(m.start())
    if len(posisi) < 2:
        return None
    return not all(a <= b for a, b in zip(posisi, posisi[1:]))

# ---------- HITUNG OTOMATIS UNTUK SEMUA DATA ----------
hasil = []
for nama, kol in MODELS.items():
    trunc = df[kol].apply(terpotong)
    ent = [len(angka_salah(a, r)) > 0 for a, r in zip(df.input_text, df[kol])]
    punya_angka = df[kol].apply(lambda r: len(angka(r)) > 0)
    hilang = [len(huruf_hilang(a, r)) > 0 for a, r in zip(df.input_text, df[kol])]
    hasil.append({
        "Model": nama,
        "Truncation (%)": round(trunc.mean() * 100, 1),
        "Entity distortion - angka/tahun (%)": round(pd.Series(ent)[punya_angka.values].mean() * 100, 1),
        "  n ringkasan berisi angka": int(punya_angka.sum()),
        "Huruf awal hilang (%)": round(pd.Series(hilang).mean() * 100, 1),
    })
hasil = pd.DataFrame(hasil)
print(f"Total data: {len(df)}\n")
print(hasil.to_string(index=False))

# ---------- BUAT FILE 30 SAMPEL UNTUK CEK MANUAL ----------
sampel = df.sample(n=30, random_state=42).reset_index(drop=True)
baris = []
for i, r in sampel.iterrows():
    for nama, kol in MODELS.items():
        a, s = r.input_text, r[kol]
        tanda = []
        if huruf_hilang(a, s):
            tanda.append("Huruf awal hilang: " + ", ".join(huruf_hilang(a, s)[:8]))
        asing = [w for w in kata_asing(a, s) if w not in huruf_hilang(a, s)]
        if asing:
            tanda.append("Kata tidak ada di artikel: " + ", ".join(asing[:8]))
        if angka_salah(a, s):
            tanda.append("Angka tidak ada di artikel: " + ", ".join(angka_salah(a, s)))
        for k in kalimat_curiga(a, s):
            tanda.append("Kalimat curiga: " + k[:150])
        if urutan_tahun_beda(a, s):
            tanda.append("Urutan tahun berbeda dari artikel")
        if terpotong(s):
            tanda.append("Terpotong")
        baris.append({
            "No": i + 1, "ID": r.id, "Model": nama,
            "Artikel": a if nama == "FLAN-T5" else "(sama dengan baris atas)",
            "Ringkasan": s,
            "Tanda dari kode": "\n".join(tanda),
            "Hallucination? (Ya/Tidak)": "",
            "Grammatical error? (Ya/Tidak)": "",
            "Chronological error? (Ya/Tidak)": "",
        })
pd.DataFrame(baris).to_excel("cek_error_manual_30.xlsx", index=False)
print("\nFile cek_error_manual_30.xlsx sudah dibuat.")

Artikel yang tidak ketemu: 0
Total data: 323

   Model  Truncation (%)  Entity distortion - angka/tahun (%)    n ringkasan berisi angka  Huruf awal hilang (%)
 FLAN-T5            96.9                                  0.8                         238                    0.6
IndoBART            95.4                                 12.5                         248                  100.0

File cek_error_manual_30.xlsx sudah dibuat.
